# Sarcasm Detection in Reddit Comments

Этот проект сфокусирован на обнаружении сарказма в комментариях на Реддите, используя классические техники NLP.

Цель проекта заключается в построении бейзлайна, анализе его поведения и улучшении результатов.

In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report


## Data Loading

Мы загружаем датасет с комментариями с Реддита и исследуем его структуру.

In [2]:
df = pd.read_csv('../Sarcasm_project/train-balanced-sarcasm.csv')
df.head()

,label,comment,author,subreddit,score,ups,downs,date,created_utc,parent_comment
0,0,NC and NH.,Trumpbart,politics,2,-1,-1,2016-10,2016-10-16 23:55:23,"Yeah, I get that argument. At this point, I'd ..."
1,0,You do know west teams play against west teams...,Shbshb906,nba,-4,-1,-1,2016-11,2016-11-01 00:24:10,The blazers and Mavericks (The wests 5 and 6 s...
2,0,"They were underdogs earlier today, but since G...",Creepeth,nfl,3,3,0,2016-09,2016-09-22 21:45:37,They're favored to win.
3,0,"This meme isn't funny none of the ""new york ni...",icebrotha,BlackPeopleTwitter,-8,-1,-1,2016-10,2016-10-18 21:03:47,deadass don't kill my buzz
4,0,I could use one of those tools.,cush2push,MaddenUltimateTeam,6,-1,-1,2016-12,2016-12-30 17:00:13,Yep can confirm I saw the tool they use for th...


In [3]:
df.columns

Index(['label', 'comment', 'author', 'subreddit', 'score', 'ups', 'downs',
       'date', 'created_utc', 'parent_comment'],
      dtype='object')

## Exploratory Data Analysis

Мы исследуем датасет по:

- количеству образцов

- балансу классов

- распределению длин комментариев

In [4]:
print(f"""Датасет сбалансированный:\n {df['label'].value_counts()[0], df['label'].value_counts()[1]}.
\n общий размер {df.shape[0]} по {df.shape[1]} колонок.
\n Самый длинный комментарий длинной {df['comment'].str.len().max()} символов 
\n Самый короткий — {df['comment'].str.len().min()} символов """)

Датасет сбалансированный:
 (505413, 505413).

 общий размер 1010826 по 10 колонок.

 Самый длинный комментарий длинной 10000.0 символов 

 Самый короткий — 1.0 символов 


In [5]:
print("Sarcastic examples:\n")
print(df[df['label'] == 1]['comment'].sample(3).values)

print("\nNot sarcastic examples:\n")
print(df[df['label'] == 0]['comment'].sample(3).values)

Sarcastic examples:

["The important thing is that we're smarter than them, and we can clearly articulate why that family just lost their innocent 2 year old in a tragic accident."
 "Make sure to watch the movie too, it's even better than the show!"
 'But how much did you lose?']

Not sarcastic examples:

["We ain't seen nothing yet" 'I could do some tweaking on it until then.'
 'Burn in hell you despotic fuck.']


## Data Preprocessing

Мы чистим текст путем:

- Привидения текста к нижнему регистру

- Убирания специальных символов

- Убирания излишних пробелов

*Так же из-за большого размера датасета (~1 миллион признаков),  был использован случайный сабсет размером 100 000 признаков для быстрого эксперимента.*

In [6]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [7]:
df_sample = df.sample(100000, random_state=17)
df_sample['clean_comment'] = df_sample['comment'].apply(clean_text)

In [8]:
df_sample[['clean_comment', 'comment']].head()

,clean_comment,comment
27291,good thing soros is so rich these guys must be...,"Good thing Soros is so rich, these guys must b..."
441956,im also not being able to send nzb files from ...,I'm also not being able to send NZB files from...
793783,or kanye west,Or Kanye West?
902822,fuck,Fuck.
469458,i think the suit does have some tech in it sim...,"I think the suit does have some tech in it, si..."


In [9]:
df_sample[(df_sample['clean_comment'] == '')]

,label,comment,author,subreddit,score,ups,downs,date,created_utc,parent_comment,clean_comment
75016,0,$650,WLVTrojanMan,hardwareswap,1,1,0,2016-09,2016-09-28 04:20:33,How much without the monitor?,
452451,0,^,archertom89,holdthemoan,7,7,0,2016-04,2016-04-30 07:42:57,Source. Please. I'm desperate.,
540617,1,*,TheGamingPlatypus18,buildapc,1,1,0,2015-07,2015-07-13 00:53:28,Bro just puke on your graphics card and you're...,
491675,0,4500,Mjays34,GlobalOffensive,0,0,0,2016-04,2016-04-14 15:24:22,3400,
587210,0,2023,burningdragons,pcmasterrace,1,1,0,2015-07,2015-07-29 06:11:43,PCMR Giveaway #9: Steam Link &amp; Controller....,
...,...,...,...,...,...,...,...,...,...,...,...
807020,0,?,potatoesgonnapotate0,leagueoflegends,1,1,0,2014-10,2014-10-05 11:55:32,Jennifer and Mila Kunis,
132880,1,........,donaldtrumptwat,ukpolitics,1,-1,-1,2016-11,2016-11-28 13:32:10,sounds like a nice guy,
138388,1,523456?,PiiNiiATA,DBZDokkanBattle,9,-1,-1,2016-10,2016-10-16 14:27:13,"If that 1 was a 5, I would do a summon.",
672165,0,:/,HatuutaH,SaltLakeCity,3,3,0,2015-04,2015-04-07 11:37:44,SLC's webcam falcon mom given 50/50 chance to ...,


In [10]:
def count_empty_strings(sample: pd.DataFrame, text: str):
    empty_count = (text == '').sum()
    total = len(df_sample)

    print(f"Пустых строк: {empty_count}")
    print(f"Доля: {empty_count / total:.4f}")

In [11]:
count_empty_strings(df_sample, df_sample['clean_comment'])

Пустых строк: 210
Доля: 0.0021


In [12]:
df_sample = df_sample[df_sample['clean_comment'].str.strip() != '']

In [13]:
df_sample = df_sample.reset_index(drop=True)

In [14]:
count_empty_strings(df_sample, df_sample['clean_comment'])

Пустых строк: 0
Доля: 0.0000


In [15]:
df_sample[(df_sample['clean_comment'] == '')]

,label,comment,author,subreddit,score,ups,downs,date,created_utc,parent_comment,clean_comment


## Train/Test Split

Разделяем датасет на тренировачный и тестовый.

In [16]:
X = df_sample['clean_comment']
y = df_sample['label']

X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=17
)

## Baseline Model

Используем TF-IDF векторизацию and Логистическую Регресию как бейзлайн.

In [37]:
vecotrizer = TfidfVectorizer(
    max_features=10000,
)

In [38]:
X_train_tfidf = vecotrizer.fit_transform(X_train)
X_test_tfidf = vecotrizer.transform(X_test)

In [39]:
print(X_train_tfidf.shape, X_test_tfidf.shape, sep='\n')

(79823, 10000)
(19956, 10000)


In [40]:
feature_names = vecotrizer.get_feature_names_out()
print(feature_names[:20])

['aa' 'aaa' 'aaaand' 'aampm' 'aap' 'aaron' 'ab' 'abandon' 'abandoned'
 'abc' 'abilities' 'ability' 'able' 'ableist' 'aboard' 'abomination'
 'abortion' 'abortions' 'about' 'above']


In [41]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [42]:
y_pred = model.predict(X_test_tfidf)

In [43]:
y_proba = model.predict_proba(X_test_tfidf)[:, 1]

In [44]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred),'\n')
print(classification_report(y_test,y_pred))

Accuracy: 0.6786931248747244
F1-score: 0.668698976955668 

              precision    recall  f1-score   support

           0       0.66      0.72      0.69      9888
           1       0.70      0.64      0.67     10068

    accuracy                           0.68     19956
   macro avg       0.68      0.68      0.68     19956
weighted avg       0.68      0.68      0.68     19956



## Эксперимент 1: Добавление n-грамм

Мы расширяем TF-IDF биграммами для выявления закономерностей на уровне фраз.

In [45]:
vecotrizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=5
)

In [46]:
X_train_tfidf = vecotrizer.fit_transform(X_train)
X_test_tfidf = vecotrizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
y_proba = model.predict_proba(X_test_tfidf)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred),'\n')
print(classification_report(y_test,y_pred))

Accuracy: 0.6913710162357186
F1-score: 0.6805332226775248 

              precision    recall  f1-score   support

           0       0.67      0.73      0.70      9888
           1       0.71      0.65      0.68     10068

    accuracy                           0.69     19956
   macro avg       0.69      0.69      0.69     19956
weighted avg       0.69      0.69      0.69     19956



## Эксперимент 2: Увеличение max_features

Мы увеличиваем количество признаков, чтобы обеспечить представление как униграмм, так и биграмм.

In [47]:
vecotrizer_2 = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=5
)

In [48]:
X_train_tfidf = vecotrizer_2.fit_transform(X_train)
X_test_tfidf = vecotrizer_2.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
y_proba = model.predict_proba(X_test_tfidf)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred),'\n')
print(classification_report(y_test,y_pred))

Accuracy: 0.6939266386049309
F1-score: 0.6832607342874922 

              precision    recall  f1-score   support

           0       0.68      0.73      0.70      9888
           1       0.71      0.65      0.68     10068

    accuracy                           0.69     19956
   macro avg       0.70      0.69      0.69     19956
weighted avg       0.70      0.69      0.69     19956



## Анализ Ошибок Модели

Анализируем ошибки модели, чтобы понять её ограничения.

In [49]:
results = pd.DataFrame({
    'text': X_test,
    'true': y_test,
    'pred': y_pred
})

mistakes = results[results['true'] != results['pred']]
mistakes.head(20)

,text,true,pred
60172,you dont own guns do you,0,1
77511,yes filling in that one form is quite a seriou...,1,0
85607,sounds like a straight to tv disney movie,0,1
81376,dude it is not your aim it is those pesky tick...,1,0
14218,i remember the good old days when when everyon...,1,0
94852,umm demand higher pay instead of trying to sup...,0,1
78508,no were talking about the other religious grou...,1,0
18114,they are if you move to britain especially now...,0,1
75781,they cant some other smart person,1,0
23697,im going to need update posts every minutes un...,1,0


## Результаты

| Model                     | F1-score |
|--------------------------|---------|
| Baseline                 | 0.6687  |
| + n-граммы               | 0.6805  |
| + больше признаков       | 0.6833  |

## Заключение

- TF-IDF обеспечивает надежную базовую модель
- Увеличение пространства признаков улучшило производительность
- Модель испытывает трудности с контекстно-зависимым сарказмом

Дальнейшая работа:
- Использование родительских комментариев
- Использование моделей на основе трансформеров (например, BERT)